# Feature-based behavioural clustering

In [ ]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
results_dir = Path("ext-data/output/results/sam3-hf/20260225_214929_sam3_hf")

df_feat = pd.read_parquet(results_dir / "dataset_features.parquet")
df_label = pd.read_parquet(results_dir / "dataset_labels.parquet")

df_joined = pd.merge(df_label, df_feat, on=["video_id", "bird_id", "window"])

# Select float64 feature columns
feat_cols = df_joined.select_dtypes(include="float64").columns.tolist()

PCA_cats = "behav"
# X = df_joined[feat_cols].dropna()
# X = df_joined[feat_cols][~df_joined["behav_label"] == "none"].dropna()
X = df_joined[
    (df_joined[PCA_cats] != "none") & (~df_joined[PCA_cats].str.contains(","))
][feat_cols].dropna()
labels = df_joined.loc[X.index, PCA_cats]

# Scale and fit PCA
X_scaled = StandardScaler().fit_transform(X)
pcs = PCA(n_components=2).fit_transform(X_scaled)

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    x=pcs[:, 0],
    y=pcs[:, 1],
    hue=labels,
    alpha=0.6,
    s=20,
    ax=ax,
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of tracking features coloured by behaviour group")
ax.legend(title=PCA_cats, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## K-Means clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

PCA_cats = "behav_group"
K = 3

mask = (df_joined[PCA_cats] != "none") & (~df_joined[PCA_cats].str.contains(","))
X = df_joined[mask][feat_cols].dropna()
labels = df_joined.loc[X.index, PCA_cats]

X_scaled = StandardScaler().fit_transform(X)
pcs = PCA(n_components=2).fit_transform(X_scaled)

km = KMeans(n_clusters=K, random_state=42, n_init="auto")
cluster_labels = km.fit_predict(X_scaled)

ari = adjusted_rand_score(labels, cluster_labels)
nmi = normalized_mutual_info_score(labels, cluster_labels)
vm  = v_measure_score(labels, cluster_labels)
print(f"ARI: {ari:.3f}  |  NMI: {nmi:.3f}  |  V-measure: {vm:.3f}")

# Side-by-side: K-Means clusters vs true labels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    x=pcs[:, 0], y=pcs[:, 1],
    hue=cluster_labels.astype(str),
    palette="tab10", alpha=0.6, s=20, ax=axes[0],
)
axes[0].set_title(f"K-Means clusters (K={K})")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")

sns.scatterplot(
    x=pcs[:, 0], y=pcs[:, 1],
    hue=labels,
    alpha=0.6, s=20, ax=axes[1],
)
axes[1].set_title(f"True labels ({PCA_cats})")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend(title=PCA_cats, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.suptitle(f"K-Means vs human labels  —  ARI={ari:.3f}, NMI={nmi:.3f}, V={vm:.3f}")
plt.tight_layout()
plt.show()

# Cluster × true-label crosstab
ct = pd.crosstab(pd.Series(cluster_labels, name="Cluster"), labels.rename("True label"))
fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Cluster × true label counts")
plt.tight_layout()
plt.show()